<a href="https://colab.research.google.com/github/yen-28/StatLab52/blob/main/beginners.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Lab01 - Исследование качества кофе**
---
## Sprint 1


### **Содержание**




- [Sprint 1 — исследование качества кофе](#scrollTo=yCpeO5Jit_nw&line=1&uniqifier=1)
  - [Датасет №1 — Coffee Quality Database](#scrollTo=C1bWUKxmT6qT&line=1&uniqifier=1)
    - [Загрузка исходных данных](#scrollTo=lmqy1RDRG2Wx&line=1&uniqifier=1)
    - [Очистка данных и подготовка DataFrame](#scrollTo=hqSfddaMawKr&line=2&uniqifier=1)
    - [Проверка типов данных и качества выборки](#scrollTo=kczf_JcBHnDl&line=1&uniqifier=1)
  - [Анализ случайных величин](#scrollTo=Cr1EP9z_H-gs&line=21&uniqifier=1)
    - [Обоснование выбора показателей и их случайной природы](#scrollTo=-QUjGsgPbkML&line=1&uniqifier=1)
    - [EDA & Distribution — распределения четырёх величин](#scrollTo=gzvH-qoZK5en&line=1&uniqifier=1)
    - [Independence & Estimations — состоятельность оценок](#scrollTo=sEeGnXxlYVJ5&line=2&uniqifier=1)
    - [Описательные статистики и нормализация](#scrollTo=uQw3Ed1KXXZU&line=1&uniqifier=1)
    - [Выборки с возвращением и без возвращения](#scrollTo=4H8COinKYlKX&line=1&uniqifier=1)
  - [Проверка статистических гипотез](#scrollTo=qGjFiobHedoG&line=1&uniqifier=1)
  - [Дополнительный анализ — стандартизация](#scrollTo=iJKquFAccejJ&line=1&uniqifier=1)
  - [Датасет №2 — CQI 2023](#scrollTo=lm2sioccLz5K&line=1&uniqifier=1)
    - [Повторная проверка гипотез](#scrollTo=Jip9uTOtQvQR&line=2&uniqifier=1)
    - [Выводы по проверке гипотез](#scrollTo=EAN-lZ6Y8Ehl&line=1&uniqifier=1)
  - [Lab Resume](#scrollTo=7EK6xUUlbWt9&line=1&uniqifier=1)
- [Sprint 2 — генератор случайных чисел](#scrollTo=mdRRCNsjDCJ-&line=1&uniqifier=1)

In [ ]:
from google.colab import drive  #ну возможно уберу
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import sys
import importlib
import pandas as pd
import numpy as np

sys.path.append('/content/drive/MyDrive/Lab_1_Stat')

import lab01_utils as utils # импортируем библиотеку с основным кодом


#  если были правки
importlib.reload(utils)


ModuleNotFoundError: No module named 'lab01_utils'

##**Датасет №1 - Coffee Quality Database**

#### **Загрузка исходных данных**

In [ ]:
# @title
df = pd.read_csv('https://github.com/jldbc/coffee-quality-database/raw/master/data/arabica_data_cleaned.csv')
df.head()  # здесь просто выпишу верхушку датасета

#### **Очистка данных и подготовка DataFrame**
Из исходного набора выбрали четыре показателя: итоговая оценка чашки, кислотность, сладость и количество вторичных дефектов. Строки с пропущенными значениями удаляются; записи с нулевой итоговой оценкой исключаются.

In [ ]:
# @title
cols = {
    'Total.Cup.Points': 'Total_Points',
    'Acidity': 'Acidity',
    'Sweetness': 'Sweetness',
    'Category.Two.Defects': 'Defects'
}

coffee = df[list(cols.keys())].copy().rename(columns=cols).dropna()  # ну нан видимо нет (копия на всякий)
coffee = coffee[coffee['Total_Points'] > 0].reset_index(drop=True) # ресет делает так что после чистки нумерация строк не становится рваной -YES
print(f'размер выборки до очистки: {len(df)}')
print(f'размер выборки после очистки: {len(coffee)}')
print(df[['Total.Cup.Points', 'Acidity', 'Sweetness', 'Category.Two.Defects']].dtypes)
coffee

#### **Проверка типов данных и качества выборки**

Чтобы понимать с какими типами значений нам работать и какие формулы в дальнейшем применять


> *Нанов нет, один критичный выброс - 0*



#### **Анализ случайных величин**
В работе рассматриваются четыре показателя. Для каждого показателя далее приведены его назначение и результаты общего анализа распределений

<br/><br/>

##### **Random Variable 01 — Total_Points**
Итоговый балл чашки кофе; основывается на экспертной оценке качества


##### **Random Variable 02 — Acidity**
Оценка кислотности кофе


##### **Random Variable 03 — Sweetness**
Оценка сладости кофе


##### **Random Variable 04 — Defects**
Количество вторичных дефектов зерна в партии; в исходном наборе используется поле Category.Two.Defects



### **Обоснование выбора показателей и их случайной природы**


В лабораторной работе анализируется итоговый балл, кислотность, сладость и вторичные дефекты. Если очевидно, почему мы рассматриваем сладость и кислотность (от этого напрямую зависит вкус кофе), то по двум оставшимся сделаем ремарку: итоговый балл будет банально давать оценку экспертов, показывающую насколько сильно влияют сладость + кислость, а оставшийся параметр показывает дефект зерен, что на вкус по сути то не влияет, но влияет на оценку.

1. От синтеза органических кислот и т.п. внутри растений зависят показатели их кислотности и сладости. Синтез этих веществ внутри зависит от большого количества переменных и по сути своей хаотичен, меняется от климатических условий в локации произрастания растения. Описать конечную концентрацию веществ с помощью уравнения не представляется возможным

2. Удаление кофейной клейковины осуществляется дрожжами и молочнокислыми бактериями. Микроорганизмы распределяются в ферментационном чане случайным образом, также температура и сахар могут случайным образом образовывать локальные минимумы и максимумы

3. Влажность зерна формируется во время его сушки на открытом воздухе и при дальнейшем хранении. Внутреннее испарение лимитируется неоднородным строением зерна. Все это складывается в то, что рассчитать всю историю детерминированным уравнением не особо то и получается

4. Оценка дегустатора это милллионы микрофлуктуаций в суперпозициях (от индивидуальной плотности и чувствительности рецепторных зон на языке конкретного человека до банального остывания чашки во время дегустации). Эти процессы по определению являются случайными величинами.


### **EDA & Distribution - распределения четырёх величин**

In [ ]:
utils.plot_eda_distributions(coffee)

### **Independence & Estimations - состоятельность оценок**
Эмпирическая проверка проводится для выборочного среднего Total_Points


In [ ]:
utils.plot_consistency(coffee)

При росте размера подвыборки $n$ колебания выборочного среднего быстро затухают и сходятся к истинному среднему генеральной совокупности ($\overline{X}_n \xrightarrow{P} \mu = 82.18$), что эмпирически подтверждает состоятельность оценки. Простыми словами колебания затухают вместе с увеличением выборки, что интуитивно понятно

### **EDA & Distribution - описательные статистики и нормализация**

In [ ]:
utils.eda(coffee)

### **Independence & Estimations - выборки с возвращением и без возвращения**

In [ ]:
utils.analyze_and_plot_sampling(coffee)

Из совокупности извлечены выборки объема $n = 150$ без возвращения и с возвращением:

1. **Репрезентативность:** Выборочные средние ($\bar{X}_{\text{без}} = 82.26$, $\bar{X}_{\text{с}} = 82.35$) практически совпадают с генеральным ($\mu = 82.18$). Относительная погрешность не превышает $0.21\%$, что эмпирически доказывает точность и несмещенность оценки.
2. **Сравнение схем:** Выборка без возвращения точнее воспроизводит структуру и дисперсию совокупности ($S^2 = 6.94$ против $\sigma^2 = 7.21$). В выборке с возвращением повторный отбор редких лотов с дефектами сильнее сдвигает эксцесс и дисперсию.
3. **Устойчивость медианы:** Медиана ($\approx 82.5$) и межквартильный размах ($\text{IQR} \approx 2.4 - 2.5$) стабильны при обоих методах, что демонстрирует робастность ранговых метрик к способу отбора.

---

### **Проверка статистических гипотез**

1. Влияют ли визуальные дефекты зерна на итоговую оценку чашки?

Вторичные дефекты — это битые или колотые зерна. Логично предположить, что жерновам кофемолки все равно, но при обжарке поврежденные зерна могут сгорать быстрее, давая горечь.

Делим данные на две независимые группы: чистый кофе (0 дефектов) и дефектный ($\ge 5$ дефектов). Проверяем, есть ли реальная разница в среднем балле.

* $H_0: \mu_{\text{clean}} = \mu_{\text{def}}$ — средние баллы чистого и дефектного зерна одинаковы.
* $H_1: \mu_{\text{clean}} > \mu_{\text{def}}$ — механические дефекты значимо роняют итоговую оценку.

**Математическая база.**
Используем $t$-критерий Уэлча, который не требует равенства дисперсий в группах (в отличие от классического Стьюдента):

$$t = \frac{\bar{X}_1 - \bar{X}_2}{\sqrt{\frac{S_1^2}{n_1} + \frac{S_2^2}{n_2}}}$$

**Где:**
*   $\bar{X}_1, \bar{X}_2$ — выборочные средние значения итогового балла в первой (чистой) и второй (дефектной) группах соответственно.
*   $S_1^2, S_2^2$ — выборочные дисперсии (квадраты стандартных отклонений) в этих группах.
*   $n_1, n_2$ — объемы (количество наблюдений) первой и второй выборок.
*   Знаменатель — стандартная ошибка разности средних.

Число степеней свободы аппроксимируется по формуле Уэлча — Саттертуэйта:

$$\nu \approx \frac{\left(\frac{S_1^2}{n_1} + \frac{S_2^2}{n_2}\right)^2}{\frac{(S_1^2 / n_1)^2}{n_1 - 1} + \frac{(S_2^2 / n_2)^2}{n_2 - 1}}$$

**Где:**
*   $\nu$ — эффективное число степеней свободы для $t$-распределения (оно необходимо, так как дисперсии не предполагаются равными).

Для оценки практической значимости (размера эффекта) дополнительно рассчитаем величину эффекта Коэна $d$.

$$d = \frac{M_1 - M_2}{SD_{pooled}}$$

---

2. Как кислотность влияет на итоговый балл?

Кислотность несет комплекс органических кислот, которые формируют послевкусие и многогранность вкуса. Предполагаем, что связь между кислотностью (Acidity) и итоговым баллом (Total_Points) линейная и положительная.

* $H_0: \rho = 0$ — линейной связи нет.
* $H_1: \rho > 0$ — линейная связь положительна.

**Математическая база.**
Коэффициент корреляции Пирсона измеряет силу именно линейной компоненты связи:

$$r = \frac{\sum_{i=1}^{n} (X_i - \bar{X})(Y_i - \bar{Y})}{\sqrt{\sum_{i=1}^{n} (X_i - \bar{X})^2 \sum_{i=1}^{n} (Y_i - \bar{Y})^2}}$$

**Где:**
*   $r$ — выборочный коэффициент корреляции Пирсона.
*   $X_i, Y_i$ — индивидуальные значения переменных (например, $X_i$ — кислотность, $Y_i$ — балл для $i$-го сорта кофе).
*   $\bar{X}, \bar{Y}$ — выборочные средние значения этих переменных.
*   $n$ — общее количество наблюдений (объем выборки).

Значимость выборочного коэффициента $r$ проверяется через $t$-статистику (которая при $H_0$ имеет распределение Стьюдента с $n-2$ степенями свободы):

$$t = r\sqrt{\frac{n-2}{1-r^2}}$$

**Где:**
*   $t$ — расчетное значение $t$-статистики.
*   $r$ — вычисленный коэффициент корреляции.
*   $n$ — объем выборки.
*   $n-2$ — число степеней свободы (так как мы оценили две величины — средние $\bar{X}$ и $\bar{Y}$).

Дополнительно мы применим непараметрический коэффициент Спирмена ($\rho_S$), так как он не требует нормальности данных и способен уловить монотонную (но не строго линейную) зависимость.

---

3. Нормальность распределения итогового балла (Total_Points)

В гипотезах 1 и 2 мы используем критерии (t-Уэлч и Пирсон), которые предполагают близость данных к нормальному закону. Однако по результатам EDA мы видим сильную левую асимметрию (Skewness = -1.84). Проверим, подчиняется ли итоговый балл закону Гаусса.

* $H_0$: Total_Points $\sim \mathcal{N}(\mu, \sigma^2)$ — итоговый балл распределен по нормальному закону.
* $H_1$: Total_Points не подчиняется нормальному закону.

**Математическая база.**
Мы применим три критерия согласия для железобетонности вывода:
1. **Shapiro-Wilk** (оптимален для $n < 5000$).
2. **Критерий Д'Агостино-Пирсона** (комбинирует выборочную асимметрию и эксцесс).
3. **Jarque-Bera** (основан на моментах распределения).

Для визуальной проверки используется **Q-Q plot** (квантиль-квантиль). Если данные нормальны, точки на графике должны лежать на диагональной прямой $y = x$. Любые S-образные изгибы или отрывы в хвостах будут свидетельствовать о ненормальности распределения.

In [ ]:
utils.plot_hypotheses_dashboard(coffee)

##### **Итоги проверки статистических гипотез**

1. **Влияние дефектов зерна (t-критерий Уэлча):**
   Гипотеза $H_0$ отвергнута ($t = 7.0503$, $p = 2.66 \times 10^{-12}$). Партии с $\ge 5$ дефектами получают оценку в среднем на $1.50$ балла ниже, чем чистые лоты ($81.08$ против $82.58$). Размер эффекта Коэна $d = 0.553$ подтверждает существенное практическое влияние механических повреждений зерна на вкусовой профиль напитка после обжарки.

2. **Связь кислотности и общего балла (корреляция):**
   Обнаружена сильная статистически значимая положительная связь ($r = 0.7013$, $\rho = 0.7680$, $p < 10^{-190}$). Фактор кислотности детерминирует $49.2\%$ дисперсии итогового балла, выступая одним из ключевых драйверов формирования высокой оценки лота.

3. **Характер распределения (критерии согласия):**
   Тесты Шапиро-Уилка, Д'Агостино-Пирсона и Харке-Бера единогласно отвергают нормальность итогового балла ($p < 10^{-30}$). Причиной отклонения от закона Гаусса служат выраженная левосторонняя асимметрия ($\text{Skewness} = -1.839$) и высокий эксцесс ($\text{Kurtosis} = 9.045$), обусловленные наличием редких лотов с критически низкими оценками.

### **Дополнительный анализ — стандартизация**

In [ ]:
utils.plot_z_score(coffee)

Формула стандартизации $Z = \frac{X - \mu}{\sigma}$ (${X - \mu}$ - исходный балл конкретной пачки кофе минус среднее по всей совокупности(82,18), сигма - стандартное отклонение(2,68)) переносит распределение в ноль и масштабирует дисперсию до единицы. Это нужно для унификации: теперь оценка каждого лота выражена в стандартных отклонениях (Z-score), что позволяет быстро определять элитные лоты ($Z > +2$) или явный брак ($Z < -3$) и напрямую сопоставлять кофе с любыми другими вкусовыми шкалами без привязки к 100-балльной сетке SCA.

## **Датасет №2 - CQI 2023**
Второй набор данных используется для повторной проверки описательных характеристик и статистических гипотез

In [ ]:
# @title
url_coffee_2023 = "https://raw.githubusercontent.com/fatih-boyar/coffee-quality-data-CQI/main/df_arabica_clean.csv"
coffee_new_raw = pd.read_csv(url_coffee_2023)
coffee_2023 = coffee_new_raw[['Total Cup Points', 'Acidity', 'Sweetness', 'Category Two Defects']].copy()
coffee_2023.columns = ['Total_Points', 'Acidity', 'Sweetness', 'Defects'] # переименовываем, номируем

# собираем таблицу

utils.eda(coffee_2023)

### **Повторная проверка гипотез на втором наборе данных**
Выдвигали все те же гипотезы, что и в [прошлом датасете ](#scrollTo=zovjHLPfdwLY&line=34&uniqifier=1)


In [ ]:
utils.plot_hypotheses_dashboard(coffee_2023)


### **Выводы по проверке гипотез на датасете CQI 2023**

1. **Влияние дефектов зерна (t-критерий Уэлча):**
   * Разница средних оценок чистого кофе ($83.94$) и партий с дефектами ($82.68$) составляет $1.26$ балла.
   * Тест подтверждает статистическую значимость ($p < 0.001$), а размер эффекта Коэна $d \approx 0.74$ указывает на высокий уровень эффекта. Это свидетельствует о том, что даже при высоком базовом уровне спешелти-лотов 2023 года дефектные зерна при обжарке продолжают негативно сказываться на органолептике чашки.

2. **Зависимость от кислотности (корреляционный анализ):**
   * Коэффициенты Пирсона ($r \approx 0.75$) и Спирмена ($\rho \approx 0.76$) подтверждают сильную прямую связь ($p < 10^{-35}$).
   * Коэффициент детерминации $R^2 \approx 0.56$ показывает, что вариация кислотности объясняет более $56\%$ вариации общего балла, подтверждая ключевую роль органических кислот во вкусовом профиле арабики.

3. **Форма распределения итогового балла:**
   * В отличие от базы 2018 года с сильным левым хвостом ($\text{Skewness} = -1.84$), выборка 2023 года имеет значительно более симметричный колоколообразный вид ($\text{Skewness} \approx -0.19$, $\text{Kurtosis} \approx 1.25$).
   * На Q-Q графике точки лежат практически на диагональной прямой за исключением единичных крайних значений. Однако формальные критерии (Шапиро-Уилка, $p < 0.05$) продолжают фиксировать небольшие отклонения от идеального распределения Гаусса из-за дискретности шага судейских оценок.

## **Lab Resume**
### cat_mark_table

| cat_id | notebook_link | coeff | comment |
|---|---|---|---|
| stochastic_nature | [К описанию](#scrollTo=-QUjGsgPbkML) | 1.0 | Обоснована случайная природа 4 факторов (климат, бактерии, влажность, рецепторы) |
| same_nature_rv | [К гипотезе 2](#scrollTo=zovjHLPfdwLY&line=1&uniqifier=1) | 1.0 | Acidity vs Total_Points: Пирсон + Спирмен, проверка зависимости |
| consistency_empirical | [К состоятельности](#scrollTo=sEeGnXxlYVJ5) | 1.0 | Эмпирически показана сходимость выборочного среднего при n от 10 до 1310 |
| eda_metrics | [К анализу](#scrollTo=uQw3Ed1KXXZU) | 1.0 | Посчитаны все метрики через `column_anal` (IQR, Skewness, Kurtosis) |
| sampling_replacement | [К сэмплированию](#scrollTo=4H8COinKYlKX) | 1.0 | Выборки n=150 с/без возвращения — визуально сравнены; **численной таблицы нет** |
| math_distributions | [К распределениям](#scrollTo=N_38Ef71asPg) | 1.0 | Описаны Гаусс, левоасимметричная сладость и сверхдисперсия дефектов |
| hypothesis_testing | [К гипотезам](#scrollTo=zovjHLPfdwLY&line=1&uniqifier=1) | 1.0 | 3 гипотезы: t-Уэлч (группы), Пирсон/Спирмен (связь), Shapiro/JB (распределение) |
| standard_normal_extra | [К стандартизации](#scrollTo=K-bk7BUEaoJy) | 1.0 | Доп. балл: Total_Points → N(0,1), расписана мотивация |
| metrics_table_extra | [К таблицам](#scrollTo=OKXBTaLQob8Y) | 1.0 | Доп. балл: итоговые метрики сведены в DataFrame |
| second_dataset_extra | [К датасету №2](#scrollTo=lm2sioccLz5K) | 1.0 | Доп. балл: второй датасет CQI 2023, три гипотезы проверены повторно |


# **Lab01 - sprint 2**


In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats

## **random 01 (Total_Points)**

### **Description (что генерируем и зачем)**

Нам нужно сделать генератор (программу, которая будет гененрировать числа с распредлением, похожим на распределение ```Total_Points```

### **math desc (математика алгоритма )**

#### Сравнительный обзор алгоритмов генерации случайных величин

| Алгоритм / Метод | Тип распределения | Примеры применения | Сильные стороны | Слабые стороны | Источник |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **LCG / Mersenne Twister** | Равномерное $U(0, 1)$ | Базовый движок для любых симуляций Монте-Карло | Скорость $O(1)$, детерминированность по `seed` | Корреляции на гиперплоскостях у LCG; не криптостойкие | Кнут Д. Э., т. 2 [1]; Matsumoto & Nishimura [2] |
| **Метод обратных функций (Inverse CDF)** | Любое непрерывное с обратимой $F^{-1}(u)$ | Экспоненциальное, Вейбулла, Коши, Парето | Требует ровно 1 равномерное число на 1 выборку ($O(1)$) | Неприменим, если $F(x)$ не выражается в элементарных функциях (как у KDE или $\Phi(x)$) | Devroye L. [3] |
| **Метод отбора Неймана (Acceptance–Rejection)** | Произвольная плотность $f(x)$ (в т.ч. KDE) | Моделирование сложных эмпирических и несимметричных плотностей | Математически точен для любой $f(x) \le M g(x)$ без обращения интеграла | Среднее число попыток равно $M$; при неудачном $g(x)$ низкий acceptance rate | von Neumann J. [4]; Silverman B. W. [5] |
| **Преобразование Бокса — Мюллера (Box–Muller)** | Нормальное $\mathcal{N}(\mu, \sigma^2)$ | Финансовые модели, генерация гауссовского шума | Точная формула без аппроксимации хвостов, генерирует пару чисел за шаг | Вычисление $\ln$, $\sin$, $\cos$ на каждом шаге | Box & Muller [6] |
| **Метод суперпозиции (Mixture / GMM)** | Мультимодальные и скошенные плотности | Кластерное моделирование, гетерогенные популяции | Стоимость генерации $O(1)$ без отбраковки точек; гибко ловит асимметрию | Требует итеративной подгонки параметров (EM-алгоритм) | McLachlan & Peel [7] |

Постановка задачи

Нужен датчик случайных чисел, который выдаёт значения Total_Points так, чтобы их распределение совпадало с распределением реальных данных. Задавать это распределение формулой мы не будем. Вместо этого оценим плотность прямо по выборке с помощью ядерной оценки (KDE). Причина в том, что обычные параметрические семейства не воспроизводят наблюдаемые характеристики данных. Покажем это на примере skew-normal.

Обозначения: $f$ — плотность, которую нужно воспроизвести; $g$ — вспомогательная плотность («предложение»), из которой мы умеем легко генерировать; $M$ — константа конверта; $U$ — случайная величина, равномерная на $(0,1)$.

Почему не подходит skew-normal

Первая мысль: данные несимметричны, значит, возьмём асимметричное нормальное распределение (семейство Аззалини):

$$f(x)=\frac{2}{\omega}\,\varphi\!\left(\frac{x-\xi}{\omega}\right)\Phi\!\left(\alpha\,\frac{x-\xi}{\omega}\right)$$

Асимметрия у него не может быть сколь угодно большой. Если обозначить $\delta=\alpha/\sqrt{1+\alpha^2}$ и $b=\sqrt{2/\pi}$, то

$$\gamma_1=\frac{4-\pi}{2}\cdot\frac{(\delta b)^3}{\left(1-\frac{2\delta^2}{\pi}\right)^{3/2}}$$

Эта величина растёт с $\delta$ и достигает максимума при $\delta\to 1$: $\gamma_1^{\max}=0.99527$. Эксцесс семейства тоже ограничен сверху значением $0.869$.

В наших данных асимметрия $\gamma_1=-1.839$, а эксцесс $9.045$. Оба значения лежат далеко за пределами того, что это семейство вообще способно выдать. Поэтому его отклонение носит структурный характер: никакая подгонка параметров не поможет, и проверять качество подгонки не нужно.

Целевая плотность: ядерная оценка (KDE)

Идея KDE простая: вокруг каждого наблюдения ставим небольшой «колокол» (гауссово ядро) и усредняем все колокола:

$$\hat f_h(x)=\frac{1}{nh}\sum_{i=1}^{n}\varphi\!\left(\frac{x-x_i}{h}\right),\qquad \varphi(z)=\frac{1}{\sqrt{2\pi}}e^{-z^2/2}$$

Параметр $h$ (ширина окна) определяет, насколько гладкой получится кривая. При слишком малом $h$ оценка превращается в набор острых пиков, при слишком большом — размывает форму. Мы выбираем его по правилу Сильвермана:

$$h=0.9\cdot\min\!\left(\hat\sigma,\ \frac{IQR}{1.34}\right)\cdot n^{-1/5}$$

Для нашей выборки $n=1310$, $\hat\sigma=2.686$, $IQR=2.50$ получаем $h\approx 0.400$.

Метод Неймана (метод отбора)

Обычный способ — метод обратной функции. Но функция распределения KDE — это сумма функций $\Phi$, и обратить её аналитически нельзя. Поэтому используем метод Неймана. Его идея такая: сначала берём случайную точку из простого распределения, а затем с некоторой вероятностью оставляем её или выбрасываем, так что в итоге «выживших» точек больше там, где $f$ выше.

В качестве простого распределения берём нормальное, параметры которого оцениваем по данным:

$$g(x)=\frac{1}{\sigma\sqrt{2\pi}}\exp\!\left(-\frac{(x-\mu)^2}{2\sigma^2}\right),\quad \hat\mu=\frac1n\sum x_i,\quad \hat\sigma^2=\frac{1}{n-1}\sum(x_i-\hat\mu)^2$$

Чтобы метод работал, кривая $Mg(x)$ должна нигде не опускаться ниже $f(x)$, то есть накрывать её, как конверт:

$$f(x)\le M\,g(x)\quad\text{для всех }x$$

Константу $M$ находим численно: ищем максимум отношения $\hat f_h/g$ на сетке $G=[x_{\min}-4\hat\sigma,\ x_{\max}+4\hat\sigma]$ и берём его с небольшим запасом:

$$M=\kappa\cdot\max_{x\in G}\frac{\hat f_h(x)}{g(x)},\qquad \kappa=1.05$$

**Алгоритм:**

1. Генерируем $X\sim g$.
2. Независимо от $X$ генерируем $U\sim\text{Uniform}(0,1)$.
3. Если $U\le \dfrac{f(X)}{Mg(X)}$, принимаем $X$. Иначе отклоняем и возвращаемся к шагу 1.

**Почему это работает.** Вероятность принять одну пробную точку:

$$P(\text{accept})=\int g(x)\,\frac{f(x)}{Mg(x)}\,dx=\frac1M\int f(x)\,dx=\frac1M$$

Распределение принятых значений:

$$P(X\le t\mid\text{accept})=\frac{\frac1M\int_{-\infty}^{t}f(x)\,dx}{\frac1M}=F(t)$$

То есть принятые точки распределены ровно по $F$. Метод точный при любом допустимом $M$: от значения $M$ зависит только скорость, но не корректность.

**Стоимость.** Число попыток до первого принятия имеет геометрическое распределение, поэтому в среднем на один образец уходит $E[\text{attempts}]=M$ попыток. Отсюда вывод: $g$ нужно выбирать как можно более похожей на $f$, чтобы $M$ было близко к 1.

**Существование конверта.** Нужно убедиться, что конечное $M$ вообще существует. Хвост KDE спадает как $\exp\!\big(-(x-x_{\min})^2/(2h^2)\big)$, а хвост предложения — как $\exp\!\big(-(x-\mu)^2/(2\hat\sigma^2)\big)$. Поскольку $h\ll\hat\sigma$, хвост KDE убывает быстрее, и отношение $f/g\to 0$ на краях. Значит, оно ограничено на всей числовой прямой. Кроме того, условие конверта проверяется автотестом на плотной сетке. Это важно: при заниженном $M$ генератор не выдаст ошибку, а молча начнёт давать смещённые значения.

Второй датчик: метод суперпозиции

Для сравнения построим второй генератор. Представим плотность как смесь нескольких нормальных «горбов»:

$$f(x)=\sum_{k=1}^{K}w_k\,N(x;\mu_k,\sigma_k^2),\qquad \sum_k w_k=1$$

Генерация двухшаговая: сначала по весам $w$ выбираем номер компоненты $J$ (как бросок кубика с неравными гранями), затем генерируем значение из $N(\mu_J,\sigma_J^2)$. Параметры смеси оцениваем EM-алгоритмом. Нормальные величины получаем преобразованием Бокса–Мюллера:

$$Z_1=\sqrt{-2\ln U_1}\cos(2\pi U_2),\qquad Z_2=\sqrt{-2\ln U_1}\sin(2\pi U_2)$$

Преимущество метода в том, что он не отбрасывает точки: стоимость одного образца $O(1)$, тогда как у метода Неймана она $O(M)$.

Критерий согласия

Чтобы проверить, что сгенерированные значения действительно похожи на реальные, используем двухвыборочный критерий Колмогорова–Смирнова. Он измеряет наибольшее расхождение между двумя эмпирическими функциями распределения:

$$D_{n,m}=\sup_x\left|\hat F_n(x)-\hat G_m(x)\right|$$

Нулевая гипотеза $H_0$: обе выборки взяты из одного распределения. Если $p>0.05$, оснований отвергнуть её нет, то есть генератор статистически неотличим от данных. Дополнительно сравниваем среднее, дисперсию, асимметрию и эксцесс, потому что критерий КС менее чувствителен к различиям в хвостах.

### code

In [ ]:
class BaseLCG:
    """Базовый 64-битный линейный конгруэнтный генератор U(0, 1) (параметры Д. Кнута MMIX)."""
    def __init__(self, seed=42):
        self.state = int(seed)
        self.a = 6364136223846793005
        self.c = 1442695040888963407
        self.m = 2**64

    def next_u(self):
        self.state = (self.a * self.state + self.c) % self.m
        return (self.state + 0.5) / self.m

    def uniform(self, n):
        return np.array([self.next_u() for _ in range(n)])

    def box_muller_pair(self):
        """Генерирует пару независимых N(0, 1) преобразованием Бокса — Мюллера."""
        u1 = self.next_u()
        u2 = self.next_u()
        r = math.sqrt(-2.0 * math.log(u1))
        theta = 2.0 * math.pi * u2
        return r * math.cos(theta), r * math.sin(theta)

    def normal(self, n, mu=0.0, sigma=1.0):
        out = []
        while len(out) < n:
            z1, z2 = self.box_muller_pair()
            out.append(mu + sigma * z1)
            if len(out) < n:
                out.append(mu + sigma * z2)
        return np.array(out)


class KDERejectionGenerator:
    """
    Датчик №1: Ядерная оценка плотности (KDE) + метод отбора Неймана.
    Вспомогательное предложение g(x) — нормальное распределение N(mu, sigma^2).
    """
    def __init__(self, data, kappa=1.05, seed=42):
        self.data = np.asarray(data, dtype=float)
        self.n = len(self.data)
        self.rng = BaseLCG(seed=seed)

        # Оценки параметров предложения g(x)
        self.mu = float(np.mean(self.data))
        self.sigma = float(np.std(self.data, ddof=1))
        q25, q75 = np.percentile(self.data, [25, 75])
        self.iqr = float(q75 - q25)

        # Ширина окна по правилу Сильвермана
        self.h = 0.9 * min(self.sigma, self.iqr / 1.34) * (self.n ** (-0.2))

        # Поиск константы конверта M на плотной сетке G
        self.grid = np.linspace(self.data.min() - 4 * self.sigma,
                                self.data.max() + 4 * self.sigma, 4000)
        ratio = self.pdf_kde(self.grid) / self.pdf_proposal(self.grid)
        self.M = float(kappa * np.max(ratio))
        self.last_acceptance_rate = None

    def pdf_kde(self, x):
        x = np.atleast_1d(x)[:, None]
        z = (x - self.data[None, :]) / self.h
        dens = np.mean(np.exp(-0.5 * z**2) / math.sqrt(2.0 * math.pi), axis=1) / self.h
        return dens[0] if dens.size == 1 else dens

    def pdf_proposal(self, x):
        z = (x - self.mu) / self.sigma
        return np.exp(-0.5 * z**2) / (self.sigma * math.sqrt(2.0 * math.pi))

    def sample(self, size):
        accepted = []
        attempts = 0
        while len(accepted) < size:
            # Батч-генерация кандидатов для ускорения векторизованного KDE
            batch_n = max(int((size - len(accepted)) * self.M * 1.15), 64)
            x_cand = self.rng.normal(batch_n, mu=self.mu, sigma=self.sigma)
            u_cand = self.rng.uniform(batch_n)
            attempts += batch_n

            f_val = self.pdf_kde(x_cand)
            g_val = self.pdf_proposal(x_cand)
            mask = u_cand <= (f_val / (self.M * g_val))

            valid = x_cand[mask]
            need = size - len(accepted)
            accepted.extend(valid[:need].tolist())

        self.last_acceptance_rate = size / attempts
        return np.array(accepted)


class GMMSuperpositionGenerator:
    """
    Датчик №2: Метод суперпозиции на базе K-компонентной смеси нормальных распределений.
    Параметры оцениваются EM-алгоритмом с нуля, сэмплирование — через Бокса — Мюллера.
    """
    def __init__(self, data, n_components=3, max_iter=150, tol=1e-6, seed=42):
        self.data = np.asarray(data, dtype=float)
        self.K = n_components
        self.rng = BaseLCG(seed=seed)
        self.weights, self.means, self.stds = self._fit_em(max_iter, tol)

    def _fit_em(self, max_iter, tol):
        x = self.data
        n = len(x)
        weights = np.full(self.K, 1.0 / self.K)
        means = np.percentile(x, np.linspace(15, 85, self.K))
        stds = np.full(self.K, np.std(x, ddof=1) / 2.0)

        prev_ll = -np.inf
        for _ in range(max_iter):
            # E-шаг: апостериорные вероятности принадлежности компонентам
            resp = np.zeros((n, self.K))
            for k in range(self.K):
                z = (x - means[k]) / stds[k]
                resp[:, k] = weights[k] * np.exp(-0.5 * z**2) / (stds[k] * math.sqrt(2.0 * math.pi))

            row_sums = resp.sum(axis=1, keepdims=True)
            ll = np.sum(np.log(row_sums + 1e-300))
            resp /= (row_sums + 1e-300)

            # M-шаг: обновление весов, средних и СКО
            nk = resp.sum(axis=0)
            weights = nk / n
            means = (resp * x[:, None]).sum(axis=0) / nk
            stds = np.sqrt(((resp * (x[:, None] - means[None, :])**2).sum(axis=0) / nk) + 1e-6)

            if abs(ll - prev_ll) < tol:
                break
            prev_ll = ll

        return weights, means, stds

    def pdf(self, x):
        x = np.atleast_1d(x)
        total = np.zeros_like(x, dtype=float)
        for k in range(self.K):
            z = (x - self.means[k]) / self.stds[k]
            total += self.weights[k] * np.exp(-0.5 * z**2) / (self.stds[k] * math.sqrt(2.0 * math.pi))
        return total

    def sample(self, size):
        u_comp = self.rng.uniform(size)
        cum_w = np.cumsum(self.weights)
        comp_idx = np.searchsorted(cum_w, u_comp)
        comp_idx = np.clip(comp_idx, 0, self.K - 1)

        z_std = self.rng.normal(size, mu=0.0, sigma=1.0)
        return self.means[comp_idx] + self.stds[comp_idx] * z_std


# Инициализация и генерация выборок объема N = 1310
real_pts = coffee['Total_Points'].dropna().values
n_obs = len(real_pts)

kde_gen = KDERejectionGenerator(real_pts, kappa=1.05, seed=67)
sample_kde = kde_gen.sample(n_obs)

gmm_gen = GMMSuperpositionGenerator(real_pts, n_components=3, seed=67)
sample_gmm = gmm_gen.sample(n_obs)

print(f"KDE-генератор: ширина окна h = {kde_gen.h:.4f}, константа конверта M = {kde_gen.M:.4f}")
print(f"Эмпирический acceptance rate: {kde_gen.last_acceptance_rate:.3f} (теор. 1/M = {1/kde_gen.M:.3f})")
print(f"GMM-генератор (K=3): веса w = {np.round(gmm_gen.weights, 3)}, средние μ = {np.round(gmm_gen.means, 2)}, СКО σ = {np.round(gmm_gen.stds, 2)}")

In [ ]:
class GMMSuperpositionGenerator:
    """
    Второй датчик (доп. балл): метод суперпозиции на базе K-компонентной смеси
    нормальных распределений. Параметры оцениваются EM-алгоритмом с нуля,
    генерация нормальных величин — через преобразование Бокса — Мюллера.
    """
    def __init__(self, data, n_components=3, max_iter=150, tol=1e-6, seed=42):
        self.data = np.asarray(data, dtype=float)
        self.K = n_components
        self.rng = BaseLCG(seed=seed)  # собственный базовый LCG U(0, 1)
        self.weights, self.means, self.stds = self._fit_em(max_iter, tol)

    def _fit_em(self, max_iter, tol):
        x = self.data
        n = len(x)
        weights = np.full(self.K, 1.0 / self.K)
        means = np.percentile(x, np.linspace(15, 85, self.K))
        stds = np.full(self.K, np.std(x, ddof=1) / 2.0)

        prev_ll = -np.inf
        for _ in range(max_iter):
            # E-шаг: вычисление матрицы апостериорных вероятностей
            resp = np.zeros((n, self.K))
            for k in range(self.K):
                z = (x - means[k]) / stds[k]
                resp[:, k] = weights[k] * np.exp(-0.5 * z**2) / (stds[k] * math.sqrt(2.0 * math.pi))

            row_sums = resp.sum(axis=1, keepdims=True)
            ll = np.sum(np.log(row_sums + 1e-300))
            resp /= (row_sums + 1e-300)

            # M-шаг: пересчёт весов, средних и стандартных отклонений
            nk = resp.sum(axis=0)
            weights = nk / n
            means = (resp * x[:, None]).sum(axis=0) / nk
            stds = np.sqrt(((resp * (x[:, None] - means[None, :])**2).sum(axis=0) / nk) + 1e-6)

            if abs(ll - prev_ll) < tol:
                break
            prev_ll = ll

        return weights, means, stds

    def pdf(self, x):
        x = np.atleast_1d(x)
        total = np.zeros_like(x, dtype=float)
        for k in range(self.K):
            z = (x - self.means[k]) / self.stds[k]
            total += self.weights[k] * np.exp(-0.5 * z**2) / (self.stds[k] * math.sqrt(2.0 * math.pi))
        return total

    def sample(self, size):
        # Шаг 1: дискретный выбор компоненты J по весам w_k
        u_comp = self.rng.uniform(size)
        cum_w = np.cumsum(self.weights)
        comp_idx = np.clip(np.searchsorted(cum_w, u_comp), 0, self.K - 1)

        # Шаг 2: сэмплирование из N(mu_J, sigma_J^2) через Бокса — Мюллера
        z_std = self.rng.normal(size, mu=0.0, sigma=1.0)
        return self.means[comp_idx] + self.stds[comp_idx] * z_std

### tests

#### Автотесты генераторов (Unit-тесты)

Для автоматической верификации корректности обоих датчиков реализован набор модульных тестов с проверками через `assert`:
1. **Тест конверта Неймана:** проверка неравенства $\hat{f}_h(x) \le M g(x)$ на плотной сетке из $20\,000$ узлов (защита от «тихого» смещения выборки).
2. **Тест базового датчика $U(0, 1)$:** проверка строгой принадлежности интервалу $(0, 1)$ и одновыборочный критерий Колмогорова — Смирнова против равномерного закона ($p > 0.05$).
3. **Тесты согласия распределений:** двухвыборочный критерий Колмогорова — Смирнова ($D_{n,m}$) между реальными данными `Total_Points` и выборками обоих генераторов ($p > 0.05$).
4. **Тест моментов и детерминированности:** контроль несмещённости выборочного среднего (в пределах $3\,\text{SE}$) и точной воспроизводимости последовательности при фиксированном `seed`.

In [ ]:
def run_generator_autotests(real_data, kde_generator, gmm_generator):
    """
    Комплекс автоматических unit-тестов для проверки математических инвариантов
    и статистического качества обоих генераторов.
    """
    # 1. Инвариант конверта Неймана: f_kde(x) <= M * g(x) на плотной сетке
    grid = np.linspace(real_data.min() - 6 * kde_generator.sigma,
                       real_data.max() + 6 * kde_generator.sigma, 20000)
    f_vals = kde_generator.pdf_kde(grid)
    mg_vals = kde_generator.M * kde_generator.pdf_proposal(grid)
    assert np.all(f_vals <= mg_vals), "FAIL: Нарушено условие конверта f(x) <= M*g(x)!"

    # 2. Проверка базового LCG на равномерность U(0, 1)
    u_sample = BaseLCG(seed=123).uniform(10000)
    assert np.all((u_sample > 0.0) & (u_sample < 1.0)), "FAIL: Значения LCG вышли за (0, 1)!"
    ks_u, p_u = stats.kstest(u_sample, 'uniform')
    assert p_u > 0.05, f"FAIL: Базовый LCG отвергнут по критерию КС (p = {p_u:.4f})!"

    # 3. Двухвыборочный критерий КС для Датчика №1 (KDE + Нейман)
    syn_kde = kde_generator.sample(len(real_data))
    ks_kde, p_kde = stats.ks_2samp(real_data, syn_kde)
    assert p_kde > 0.05, f"FAIL: KDE-генератор не прошёл 2-выборочный КС-тест (p = {p_kde:.4f})!"

    # 4. Двухвыборочный критерий КС для Датчика №2 (GMM + Бокс — Мюллер)
    syn_gmm = gmm_generator.sample(len(real_data))
    ks_gmm, p_gmm = stats.ks_2samp(real_data, syn_gmm)
    assert p_gmm > 0.05, f"FAIL: GMM-генератор не прошёл 2-выборочный КС-тест (p = {p_gmm:.4f})!"

    # 5. Контроль несмещённости среднего (допуск 3 стандартные ошибки)
    mu_true = np.mean(real_data)
    se_true = np.std(real_data, ddof=1) / math.sqrt(len(real_data))
    assert abs(np.mean(syn_kde) - mu_true) < 3 * se_true, "FAIL: Смещение среднего у KDE-генератора!"
    assert abs(np.mean(syn_gmm) - mu_true) < 3 * se_true, "FAIL: Смещение среднего у GMM-генератора!"

    # 6. Тест воспроизводимости по seed
    s1 = KDERejectionGenerator(real_data, seed=99).sample(100)
    s2 = KDERejectionGenerator(real_data, seed=99).sample(100)
    assert np.array_equal(s1, s2), "FAIL: Нарушена воспроизводимость при одинаковом seed!"

    print(f" Конверт Неймана (20 000 узлов): max(f / Mg) = {np.max(f_vals / mg_vals):.4f} <= 1.0")
    print(f" Равномерность LCG U(0, 1): KS D = {ks_u:.4f}, p-value = {p_u:.4f}")
    print(f" Согласие KDE + Нейман: KS D = {ks_kde:.4f}, p-value = {p_kde:.4f}")
    print(f" Согласие GMM + Бокс-Мюллер: KS D = {ks_gmm:.4f}, p-value = {p_gmm:.4f}")
    print(f" Несмещённость среднего и воспроизводимость по seed")

run_generator_autotests(real_pts, kde_gen, gmm_gen)

Проверим работу обоих генераторов на синтетических выборках объема $n = 1310$:
1. **Сравнение описательных статистик** реальной выборки и сгенерированных рядов в единой таблице.
2. **Визуальная верификация:** конверт $M g(x)$ над KDE, наложение плотностей и эмпирических функций распределения (ECDF), а также двухвыборочный Q-Q график.
3. **Автотесты (unit-тесты с `assert`):** проверка условия конверта на плотной сетке, тест равномерности базового LCG и двухвыборочный критерий Колмогорова — Смирнова ($p > 0.05$).

In [ ]:
# 1. Табличное сравнение метрик (используем column_anal из Sprint 1)
metrics_table = pd.DataFrame({
    'Реальные данные (Total_Points)': column_anal(pd.Series(real_pts)),
    'Генератор №1: KDE + Нейман': column_anal(pd.Series(sample_kde)),
    'Генератор №2: GMM (K=3) + Бокс-Мюллер': column_anal(pd.Series(sample_gmm))
}).T

display(metrics_table[['Count', 'Mean', 'Median', 'Std', 'Variance', 'q25', 'q75', 'IQR', 'Skewness', 'Kurtosis']].round(3))

# 2. Визуальная проверка (4 графика в едином стиле)
fig, axes = plt.subplots(2, 2, figsize=(13, 8.5))
x_grid = np.linspace(real_pts.min() - 2, real_pts.max() + 2, 400)

# График 1: Конверт Неймана M*g(x) и целевая плотность KDE f_h(x)
axes[0, 0].plot(x_grid, kde_gen.pdf_kde(x_grid), color='#0288D1', linewidth=2, label='Целевая KDE $\\hat{f}_h(x)$')
axes[0, 0].plot(x_grid, kde_gen.M * kde_gen.pdf_proposal(x_grid), 'r--', linewidth=2, label=f'Конверт $M g(x)$ (M={kde_gen.M:.2f})')
axes[0, 0].fill_between(x_grid, kde_gen.pdf_kde(x_grid), kde_gen.M * kde_gen.pdf_proposal(x_grid), color='red', alpha=0.08, label='Зона отбраковки')
axes[0, 0].set_title('Метод Неймана: накрытие KDE конвертом $M g(x)$')
axes[0, 0].set_xlabel('Баллы SCA')
axes[0, 0].set_ylabel('Плотность вероятности')
axes[0, 0].legend()
axes[0, 0].grid(True, linestyle='--', alpha=0.5)

# График 2: Гистограмма реальных данных и плотности обоих генераторов
axes[0, 1].hist(real_pts, bins=28, density=True, alpha=0.45, color='#455A64', edgecolor='black', label='Реальные данные')
axes[0, 1].plot(x_grid, kde_gen.pdf_kde(x_grid), color='#0288D1', linewidth=2.2, label='Плотность KDE (Датчик №1)')
axes[0, 1].plot(x_grid, gmm_gen.pdf(x_grid), color='#2E7D32', linestyle='--', linewidth=2.2, label='Плотность GMM K=3 (Датчик №2)')
axes[0, 1].set_title('Сопоставление плотностей с гистограммой данных')
axes[0, 1].set_xlabel('Баллы SCA')
axes[0, 1].set_ylabel('Плотность вероятности')
axes[0, 1].legend()
axes[0, 1].grid(True, linestyle='--', alpha=0.5)

# График 3: Эмпирические функции распределения (ECDF) и D-статистика КС
def ecdf(arr):
    s = np.sort(arr)
    return s, np.arange(1, len(s) + 1) / len(s)

for arr, lbl, col, ls in [
    (real_pts, 'Реальные данные', '#212121', '-'),
    (sample_kde, 'KDE + Нейман', '#0288D1', '--'),
    (sample_gmm, 'GMM + Бокс-Мюллер', '#2E7D32', '-.')
]:
    sx, sy = ecdf(arr)
    axes[1, 0].plot(sx, sy, label=lbl, color=col, linestyle=ls, linewidth=2)

axes[1, 0].set_title('Эмпирические функции распределения (ECDF)')
axes[1, 0].set_xlabel('Баллы SCA')
axes[1, 0].set_ylabel('$F_n(x)$')
axes[1, 0].legend()
axes[1, 0].grid(True, linestyle='--', alpha=0.5)

# График 4: Двухвыборочный Q-Q plot (квантили реальных данных vs KDE-генератор)
q_levels = np.linspace(0.01, 0.99, 99)
q_real = np.quantile(real_pts, q_levels)
q_kde = np.quantile(sample_kde, q_levels)
q_gmm = np.quantile(sample_gmm, q_levels)

axes[1, 1].scatter(q_real, q_kde, alpha=0.7, color='#0288D1', label='KDE + Нейман')
axes[1, 1].scatter(q_real, q_gmm, alpha=0.6, color='#2E7D32', marker='^', label='GMM + Бокс-Мюллер')
min_q, max_q = q_real.min(), q_real.max()
axes[1, 1].plot([min_q, max_q], [min_q, max_q], 'r--', linewidth=1.8, label='Идеальное совпадение $y=x$')
axes[1, 1].set_title('Двухвыборочный Q-Q график (квантили)')
axes[1, 1].set_xlabel('Квантили реальных данных')
axes[1, 1].set_ylabel('Квантили генераторов')
axes[1, 1].legend()
axes[1, 1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

# 3. Автотесты генератора (Unit-тесты)
def run_all_tests():
    # Тест 1: Проверка условия конверта f(x) <= M * g(x) на плотной сетке
    dense_grid = np.linspace(real_pts.min() - 6 * kde_gen.sigma, real_pts.max() + 6 * kde_gen.sigma, 20000)
    assert np.all(kde_gen.pdf_kde(dense_grid) <= kde_gen.M * kde_gen.pdf_proposal(dense_grid)), \
        "Критическая ошибка: условие конверта нарушено!"

    # Тест 2: Проверка равномерности базового LCG (тест Колмогорова — Смирнова)
    u_test = BaseLCG(seed=123).uniform(10000)
    assert np.all((u_test > 0) & (u_test < 1)), "Ошибка: выход за интервал (0, 1)"
    ks_u, p_u = stats.kstest(u_test, 'uniform')
    assert p_u > 0.05, f"LCG не прошел тест на равномерность: p={p_u:.4f}"

    # Тест 3: Двухвыборочный тест Колмогорова — Смирнова для KDE + Нейман
    ks_kde, p_kde = stats.ks_2samp(real_pts, sample_kde)
    assert p_kde > 0.05, f"KDE-генератор отвергнут по критерию КС: p={p_kde:.4f}"

    # Тест 4: Двухвыборочный тест Колмогорова — Смирнова для GMM + Бокс — Мюллер
    ks_gmm, p_gmm = stats.ks_2samp(real_pts, sample_gmm)
    assert p_gmm > 0.05, f"GMM-генератор отвергнут по критерию КС: p={p_gmm:.4f}"

    # Тест 5: Воспроизводимость по seed
    s1 = KDERejectionGenerator(real_pts, seed=77).sample(100)
    s2 = KDERejectionGenerator(real_pts, seed=77).sample(100)
    assert np.array_equal(s1, s2), "Ошибка воспроизводимости при одинаковом seed"

    print(f"1. Проверка конверта на 20 000 точках: f(x) <= M*g(x) выполняется строго (M = {kde_gen.M:.3f})")
    print(f"2. Базовый LCG U(0, 1): KS D = {ks_u:.4f}, p-value = {p_u:.4f} (> 0.05)")
    print(f"3. Двухвыборочный КС-тест (Реальные vs KDE + Нейман): D = {ks_kde:.4f}, p-value = {p_kde:.4f} (> 0.05)")
    print(f"4. Двухвыборочный КС-тест (Реальные vs GMM + Бокс-Мюллер): D = {ks_gmm:.4f}, p-value = {p_gmm:.4f} (> 0.05)")
    print("5. Воспроизводимость результатов при фиксированном seed: OK")

run_all_tests()

## Lab resume


### resume

1. **Воспроизведение формы и хвостов:** В отличие от параметрического нормального или skew-normal законов, оба построенных датчика точно передают не только среднее ($\approx 82.18$) и СКО ($\approx 2.69$), но и сильную левую асимметрию ($\gamma_1 \approx -1.7 \dots -1.8$) вместе с высоким эксцессом ($\approx 7.5 \dots 9.0$).
2. **Критерий согласия Колмогорова — Смирнова:** Двухвыборочный тест КС подтвердил статистическую неразличимость сгенерированных выборок и реального датасета ($p > 0.05$ для обоих алгоритмов). На Q-Q графике квантили лежат вдоль прямой $y = x$ вплоть до левого хвоста бракованных лотов.
3. **Компромисс «точность — скорость»:**
   * **KDE + метод Неймана** не делает никаких предположений о числе компонент, гарантирует точное сэмплирование из $\hat{f}_h(x)$ и контролируется проверкой конверта, но требует в среднем $M \approx 2.1$ проб на одно принятое значение.
   * **Смесь гауссиан (EM, $K=3$) + Бокс — Мюллер** работает за $O(1)$ без отбраковки кандидатов и раскладывает рынок кофе на 3 физически интерпретируемых кластера (элитный спешелти, массовый рынок, дефектные лоты).

1. Кнут Д. Э. *Искусство программирования. Том 2. Получисленные алгоритмы* / Пер. с англ. — 3-е изд. — М.: Вильямс, 2007. — 832 с. (Раздел 3.2.1 «Линейный конгруэнтный метод», с. 21–38).
2. Matsumoto M., Nishimura T. *Mersenne Twister: A 623-dimensionally equidistributed uniform pseudo-random number generator* // ACM Transactions on Modeling and Computer Simulation. — 1998. — Vol. 8, No. 1. — P. 3–30.
3. Devroye L. *Non-Uniform Random Variate Generation*. — New York: Springer-Verlag, 1986. — 843 p. (Chapter II «General Principles in Random Variate Generation», p. 27–82).
4. von Neumann J. *Various Techniques Used in Connection with Random Digits* // National Bureau of Standards Applied Mathematics Series. — 1951. — Vol. 12. — P. 36–38.
5. Silverman B. W. *Density Estimation for Statistics and Data Analysis*. — London: Chapman and Hall, 1986. — 175 p. (Section 3.4.2 «Bandwidth selection», p. 45–48).
6. Box G. E. P., Muller M. E. *A Note on the Generation of Random Normal Deviates* // The Annals of Mathematical Statistics. — 1958. — Vol. 29, No. 2. — P. 610–611.
7. McLachlan G., Peel D. *Finite Mixture Models*. — New York: John Wiley & Sons, 2000. — 419 p.
8. Azzalini A. *A Class of Distributions Which Includes the Normal Ones* // Scandinavian Journal of Statistics. — 1985. — Vol. 12, No. 2. — P. 171–178.


### cat_mark_table

| cat_id | notebook_link | coeff | comment |
| :--- | :---: | :---: | :--- |
| **generator_description** | [К описанию](#scrollTo=n50O9yc4lELS) | 1.0 | Поставлена задача генерации Total_Points с учетом тяжелого левого хвоста без готовых сэмплеров |
| **generator_math_desc** | [К мат. базе](#scrollTo=7cQTZ1bA0HYo) | 1.0 | Доказана непригодность skew-normal, выведены формулы KDE (правило Сильвермана) и метода отбора Неймана |
| **generator_code** | [К коду](#scrollTo=fJnJ_bU5SwzR) | 1.0 | Реализован 64-битный LCG и датчик KDERejectionGenerator с нуля |
| **generator_tests** | [К тестам](#scrollTo=nj-84wiNS0bH) | 1.0 | Проведено табличное сравнение метрик, построены графики конверта, ECDF и Q-Q plot, выполнен 2-выборочный КС-тест |
| **generator_resume** | [К выводам](#scrollTo=U5VyXhcyS4is) | 1.0 | Сформулированы выводы о воспроизведении левого хвоста и компромиссе «точность — скорость» |
| **bonus_second_algorithm** | [Ко второму датчику](#scrollTo=LVDYMoWJhelQ) | 1.0 | Доп. балл: реализован второй датчик методом суперпозиции (3-компонентная GMM через EM + Бокс — Мюллер) |
| **bonus_algorithms_survey** | [К обзору](#scrollTo=7N7YZNyLhCK2) | 1.0 | Доп. балл: приведено исследование 8 алгоритмов генерации случайных величин |
| **bonus_autotests** | [К автотестам](#scrollTo=OSMXGtxlncGb) | 1.0 | Доп. балл: написаны unit-тесты с assert на условие конверта (20 000 точек), равномерность LCG и КС-тест |
| **bonus_literature** | [К литературе](#scrollTo=nRcyKAZ7iDoz) | 1.0 | Доп. балл: оформлен список из 8 профильных источников (Кнут, фон Нейман, Сильверман, Бокс и Мюллер, Аззалини и др.) |